<a href="https://colab.research.google.com/github/niniqoiii/Springfield/blob/main/inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install -q gdown

import os
import json
import zipfile
import shutil
import random
import gdown
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms


In [17]:
file_id = "12FqLnmk6Cu0NR8UehRQiD7az2iDYufCX"

zip_file = "/content/Simpsons.zip"
extract_dir = "/content/simpsons_extracted"

if not os.path.exists(zip_file):
    gdown.download(f"https://drive.google.com/uc?id={file_id}", zip_file, quiet=False)

if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Dataset extracted")


Dataset extracted


In [18]:
train_dir = None

for root, dirs, files in os.walk(extract_dir):
    if "characters_train" in dirs and "__MACOSX" not in root:
        train_dir = os.path.join(root, "characters_train")
        break

print("Train dir:", train_dir)


Train dir: /content/simpsons_extracted/archive/characters_train


In [19]:
random.seed(42)

test_dir = "/content/test_images"

if os.path.exists(test_dir):
    shutil.rmtree(test_dir)

os.makedirs(test_dir, exist_ok=True)

classes = sorted([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])

selected_classes = random.sample(classes, min(15, len(classes)))

copied = 0

for cls in selected_classes:

    cls_path = os.path.join(train_dir, cls)

    images = [
        f for f in os.listdir(cls_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    random.shuffle(images)

    for img in images[:5]:

        src = os.path.join(cls_path, img)

        dst = os.path.join(test_dir, f"{cls}_{img}")

        shutil.copy(src, dst)

        copied += 1

print(f"Created random test_images folder with {copied} images")
print("Classes used:", selected_classes)


Created random test_images folder with 75 images
Classes used: ['troy_mcclure', 'chief_wiggum', 'agnes_skinner', 'krusty_the_clown', 'homer_simpson', 'groundskeeper_willie', 'cletus_spuckler', 'charles_montgomery_burns', 'carl_carlson', 'moe_szyslak', 'apu_nahasapeemapetilon', 'abraham_grampa_simpson', 'patty_bouvier', 'rainier_wolfcastle', 'waylon_smithers']


In [20]:
class SimpleCNN(nn.Module):

    def __init__(self, num_classes):

        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


In [21]:
def infer(data_dir, model_path):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    with open("classes.json", "r") as f:
        classes = json.load(f)

    model = SimpleCNN(len(classes))

    model.load_state_dict(torch.load(model_path, map_location=device))

    model.to(device)

    model.eval()

    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])

    results = {}

    for file in sorted(os.listdir(data_dir)):

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(data_dir, file)

            image = Image.open(path).convert("RGB")

            image = transform(image).unsqueeze(0).to(device)

            with torch.no_grad():

                outputs = model(image)

                predicted = torch.argmax(outputs, dim=1).item()

            results[file] = classes[predicted]

    with open("results.json", "w") as f:
        json.dump(results, f, indent=4)

    print(f"results.json created with {len(results)} predictions")

    return results


In [23]:
model_path = "model.pth"

results = infer(test_dir, model_path)

list(results.items())[:20]


results.json created with 75 predictions


[('abraham_grampa_simpson_pic_0001.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0304.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0335.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0440.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0722.jpg', 'homer_simpson'),
 ('agnes_skinner_pic_0000.jpg', 'homer_simpson'),
 ('agnes_skinner_pic_0003.jpg', 'homer_simpson'),
 ('agnes_skinner_pic_0014.jpg', 'homer_simpson'),
 ('agnes_skinner_pic_0016.jpg', 'homer_simpson'),
 ('agnes_skinner_pic_0026.jpg', 'homer_simpson'),
 ('apu_nahasapeemapetilon_pic_0048.jpg', 'homer_simpson'),
 ('apu_nahasapeemapetilon_pic_0055.jpg', 'homer_simpson'),
 ('apu_nahasapeemapetilon_pic_0068.jpg', 'homer_simpson'),
 ('apu_nahasapeemapetilon_pic_0288.jpg', 'nelson_muntz'),
 ('apu_nahasapeemapetilon_pic_0368.jpg', 'homer_simpson'),
 ('carl_carlson_pic_0003.jpg', 'homer_simpson'),
 ('carl_carlson_pic_0027.jpg', 'homer_simpson'),
 ('carl_carlson_pic_0048.jpg', 'homer_simpson'),
 ('carl_carlso